# Advanced — Nonlinear methods, velocity fields, cross-subject alignment

Companion to the main tutorial. Loads the artifacts produced by
`tutorial_pca_trajectories_eegbci.ipynb` and runs three showcases
that *extend* PCA (paper §5 / §6):

1. **Nonlinear methods** — PCA vs UMAP vs PHATE vs Isomap, ranked by
   `MethodSelector` on trustworthiness / continuity.
2. **Velocity fields** — `compute_velocity_fields` +
   `plot_streamlines` over PC1–PC2.
3. **Cross-subject alignment** — `flip_pc_scores_for_consistency`
   recap then Procrustes between subject-pair manifolds.

**Prerequisite:** run the main tutorial first. This notebook loads
`outputs/tutorial_eegbci/artifacts/` and will fail with a clear
message if it is absent.


In [ ]:
import sys
from pathlib import Path

_PROJECT_ROOT = Path.cwd()
if (_PROJECT_ROOT / 'coco_pipe').is_dir() is False and (_PROJECT_ROOT.parent / 'coco_pipe').is_dir():
    _PROJECT_ROOT = _PROJECT_ROOT.parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

import numpy as np
import pandas as pd

from coco_pipe.dim_reduction import DimReduction
from coco_pipe.dim_reduction.evaluation import MethodSelector, compute_velocity_fields
from coco_pipe.report import Report, Section, PlotlyElement
from coco_pipe.report.elements import InteractiveTableElement
from coco_pipe.viz.interactive import (
    plot_embedding,
    plot_shepard_diagram,
    plot_streamlines,
    plot_trajectory,
)
from tutorials._helpers import LABEL_NAMES, load_artifacts

rng = np.random.default_rng(42)
ARTIFACTS_DIR = Path('outputs/tutorial_eegbci/artifacts')
OUT = Path('outputs/tutorial_eegbci_advanced')
OUT.mkdir(parents=True, exist_ok=True)

art = load_artifacts(ARTIFACTS_DIR, method='PCA')
container_traj = art.trajectory_container
trial_labels = np.asarray(container_traj.y).astype(int)
times = np.asarray(container_traj.coords['time'])
print('loaded:', container_traj.X.shape, 'subjects:', list(art.per_subject_reducers.keys())[:5])

report = Report(title='Advanced — Nonlinear, velocity, alignment', asset_urls='inline')


## A — Nonlinear method comparison

Fit UMAP / PHATE / Isomap on the same pooled samples that the shared
PCA saw, score them with the same trustworthiness/continuity metrics,
and rank with `MethodSelector`. PHATE/Isomap are heavier; we sub-
sample to ~3k points to keep the cell fast.


In [ ]:
# Reload raw sensor-space EEGBCI so all reducers see the same high-dim
# input (the saved trajectory container only carries PC scores).
from tutorials._helpers import DEFAULT_CONDITIONS, load_eegbci_container
container_raw = load_eegbci_container(
    Path('PhysioNet_EEGBCI/BIDS'), runs=list(range(3, 15)),
    conditions=DEFAULT_CONDITIONS,
)
times_full = np.asarray(container_raw.coords['time'])
time_mask = (times_full >= 0 - 1e-8) & (times_full <= 1.5 + 1e-8)
container_z = (
    container_raw.isel(time=time_mask)
    .baseline_correction(dim='time')
    .zscore(dim='obs')
)
container_pooled_raw = container_z.stack(dims=('obs', 'time'), new_dim='obs')
X_pool_all = container_pooled_raw.X
n_sub = min(3000, X_pool_all.shape[0])
idx = rng.choice(X_pool_all.shape[0], size=n_sub, replace=False)
X_score = X_pool_all[idx]
print('comparing reducers on X with shape', X_score.shape)


In [ ]:
# Fit + score each reducer on the same subsample, then rank with MethodSelector.
methods = ['PCA', 'UMAP', 'PHATE', 'Isomap']
reducers: dict = {}
embeddings: dict = {}
scores: dict = {}
shepard_pairs: dict = {}
for method_name in methods:
    try:
        red = DimReduction(method=method_name, n_components=3, random_state=42)
        emb = red.fit_transform(X_score)
        payload = red.score(X_emb=emb, X=X_score, n_neighbors=15,
                            metrics=['trustworthiness', 'continuity'])
        reducers[method_name] = red
        embeddings[method_name] = emb
        scores[method_name] = payload.get('metrics', {})
        shepard_pairs[method_name] = (X_score, emb)
    except (ImportError, ValueError) as e:
        print(f'skip {method_name}: {e}')

summary = (
    pd.DataFrame(scores).T.reset_index().rename(columns={'index': 'method'})
)

# MethodSelector ranks by a primary metric with a tie-breaker.
ranking = None
try:
    selector = MethodSelector(reducers).collect()
    ranking = selector.rank_methods(
        selection_metric='trustworthiness', tie_breakers=['continuity'],
    )
    print(ranking)
except Exception as exc:
    print('MethodSelector ranking unavailable:', exc)
    ranking = summary.sort_values('trustworthiness', ascending=False)

sec_a = Section('A — Nonlinear comparison', icon='A')
sec_a.add_markdown(
    'Each method fit on the same 3000-sample subsample of sensor-space '
    'epochs; ranked by trustworthiness with continuity tie-breaker.'
)
sec_a.add_element(InteractiveTableElement(summary, title='Reducer quality summary'))
if ranking is not None:
    sec_a.add_element(InteractiveTableElement(
        ranking, title='MethodSelector ranking (trustworthiness, tied on continuity)'
    ))


In [ ]:
# Side-by-side Shepard diagrams (one per method) — visual check on
# whether each embedding preserves the original distance structure.
for method_name, (X_o, X_e) in shepard_pairs.items():
    # `sample_size=300` keeps each Shepard plot under ~1 MB embedded.
    fig_sh = plot_shepard_diagram(
        X_orig=X_o, X_emb=X_e, sample_size=300,
        title=f'{method_name} — Shepard diagram',
        random_state=42,
    )
    sec_a.add_element(PlotlyElement(fig_sh, height='380px'))


In [ ]:
# Side-by-side trajectory plots: project per-condition trial means into
# each method's space, then plot.
from dataclasses import replace as _replace
trial_idx = np.arange(container_traj.X.shape[0])  # one trial = (n_times, n_components)

# Per-method group-mean trajectory in the *method's* embedding space.
# Approach: for each trial, average across timepoints first (one sample),
# then project that one sample with each fitted reducer.
trial_mean_hd = container_pooled_raw.X.reshape(
    container_traj.X.shape[0], container_traj.X.shape[1], -1
).mean(axis=1)

for method_name, red in reducers.items():
    try:
        emb_trials = red.transform(trial_mean_hd)
    except Exception:
        # Some non-linear methods (UMAP, PHATE) can be expensive on new data;
        # fall back to refitting on the trial-mean set directly.
        red_proj = DimReduction(method=method_name, n_components=3, random_state=42)
        emb_trials = red_proj.fit_transform(trial_mean_hd)
    fig_emb = plot_embedding(
        embedding=emb_trials[:, :2],
        labels=np.array([LABEL_NAMES[int(c)] for c in trial_labels]),
        title=f'{method_name} — trial-mean PC1-PC2 colored by condition',
        dimensions=2,
    )
    sec_a.add_element(PlotlyElement(fig_emb, height='460px'))

report.add_section(sec_a)


## B — Velocity fields over PC1–PC2

`compute_velocity_fields` estimates per-sample velocity vectors and
`plot_streamlines` renders them as streamlines over the embedding.
Colored by condition shows where the trajectories diverge most.


In [ ]:
# High-dim X = sensor-space pooled samples (from Section A); low-dim
# embedding = the PC scores already in the saved trajectory container.
X_hd = container_pooled_raw.X
container_pc_pooled = container_traj.stack(dims=('obs', 'time'), new_dim='obs')
X_lo = container_pc_pooled.X[:, :2]  # PC1–PC2
groups = np.repeat(trial_labels, container_traj.X.shape[1])
vel = compute_velocity_fields(X_hd, X_lo, delta_t=1, n_neighbors=20)
fig_stream = plot_streamlines(
    X_emb=X_lo, V_emb=vel,
    title='B — Velocity streamlines (PC1–PC2)',
)
sec_b = Section('B — Velocity fields', icon='B')
sec_b.add_element(PlotlyElement(fig_stream, height='560px'))
report.add_section(sec_b)


## C — Cross-subject alignment

We already sign-flipped per-subject scores in the main tutorial; that
fixes sign but not basis swaps or rotations. Here we show a simple
Procrustes alignment between two subjects' per-subject PC1–PC3
centroids and visualise the *before / after* trajectories.


In [ ]:
from scipy.linalg import orthogonal_procrustes

subs = list(art.per_subject_reducers.keys())[:2]
if len(subs) < 2:
    print('Need ≥ 2 subjects in artifacts to run alignment.')
else:
    sub_a, sub_b = subs
    # Build per-subject mean trajectories over time per condition
    cont = container_traj
    sub_id_per_trial = np.asarray(cont.coords.get('subject', cont.ids))
    def mean_per_cond(sub):
        out = []
        for cond in sorted(set(trial_labels.tolist())):
            mask = (sub_id_per_trial == sub) & (trial_labels == cond)
            if not mask.any():
                out.append(None)
                continue
            out.append(cont.X[mask][..., :3].mean(axis=0))
        return out
    A = mean_per_cond(sub_a)
    B = mean_per_cond(sub_b)
    pairs = [(a, b) for a, b in zip(A, B) if a is not None and b is not None]
    if pairs:
        stacked_a = np.concatenate([p[0] for p in pairs], axis=0)
        stacked_b = np.concatenate([p[1] for p in pairs], axis=0)
        R, _ = orthogonal_procrustes(stacked_b, stacked_a)
        B_aligned = [b @ R for _, b in pairs]

        X_before = np.stack([np.stack([a, b]) for a, b in pairs])  # (n_cond, 2_subs, t, 3)
        X_after = np.stack([np.stack([a, b]) for a, (_, b) in zip([p[0] for p in pairs], zip(pairs, B_aligned))])

        fig_before = plot_trajectory(
            np.concatenate([np.stack([p[0], p[1]]) for p in pairs], axis=0),
            labels=np.array([f'cond{i}_{s}' for i in range(len(pairs)) for s in ('A', 'B')]),
            title='C — Before Procrustes alignment', dimensions=3,
        )
        fig_after = plot_trajectory(
            np.concatenate([np.stack([a, b]) for a, (_, b) in zip([p[0] for p in pairs], zip(pairs, B_aligned))], axis=0),
            labels=np.array([f'cond{i}_{s}' for i in range(len(pairs)) for s in ('A', 'B-aligned')]),
            title='C — After Procrustes alignment (B rotated to A)', dimensions=3,
        )
        sec_c = Section('C — Cross-subject alignment', icon='C')
        sec_c.add_element(PlotlyElement(fig_before, height='520px'))
        sec_c.add_element(PlotlyElement(fig_after, height='520px'))
        report.add_section(sec_c)

report.save(str(OUT / 'report_tutorial_advanced.html'))
print('saved:', OUT / 'report_tutorial_advanced.html')
